### Document Loaders in LangChain

Document Loaders ingest data from external sources (files, web pages, APIs, databases) and transform it into standardized LangChain Document objects.

Each Document object contains:
* page_content: The extracted text string payload.
* metadata: Dictionary containing source details like file path, page number, or URL.

---

### Core Loader Methods

* load(): Synchronously loads all documents at once into an in-memory list (List[Document]). Convenient for small files, but loads everything into memory.
* lazy_load(): Returns an iterator (Iterator[Document]) to stream documents one at a time. Essential for handling large files or directories with low memory footprint.
* aload(): Asynchronously loads all documents into a list.
* alazy_load(): Asynchronously streams documents lazily as an iterator.
* load_and_split(): Loads documents and splits them using a specified TextSplitter in one step.

---

### Document Loaders Reference Matrix

| Loader Category | Loader Class | Primary Source / Extension | Short Description |
| :--- | :--- | :--- | :--- |
| Base & Custom | BaseLoader | Python subclass | Abstract base class for implementing custom loaders. |
| Text & Markdown | TextLoader | .txt, .log, .md | Ingests plain text files into a single Document. |
| | UnstructuredMarkdownLoader | .md | Parses formatted Markdown text and elements. |
| | DirectoryLoader | File directory | Batch-loads directory contents using glob patterns. |
| PDF Documents | PyPDFLoader | .pdf | Fast page-by-page PDF text extraction. |
| | PyMuPDFLoader | .pdf | High-performance C-backed PDF parser. |
| | PDFMinerLoader | .pdf | Extracts layout coordinates and text positions. |
| | UnstructuredPDFLoader | .pdf | Scanned PDFs with OCR & complex layout parsing. |
| Office Documents | UnstructuredWordDocumentLoader | .docx, .doc | Microsoft Word documents. |
| | UnstructuredPowerPointLoader | .pptx | Microsoft PowerPoint slides. |
| | UnstructuredExcelLoader | .xlsx | Microsoft Excel workbooks. |
| | CSVLoader | .csv | Converts CSV rows into individual Document objects. |
| Web & Scraping | WebBaseLoader | Web URLs | Scrapes HTML web pages using BeautifulSoup4. |
| | RecursiveUrlLoader | Website domain | Recursively crawls links up to a maximum depth. |
| | SitemapLoader | sitemap.xml | Bulk-loads pages listed in sitemap XML files. |
| | ArXivLoader | arXiv IDs | Scientific papers and metadata from arXiv. |
| | WikipediaLoader | Search queries | Searches and fetches Wikipedia articles. |
| Structured & Cloud | JSONLoader | .json, .jsonl | Extracts JSON fields using jq expressions. |
| | GenericLoader | .py, .js, .cpp | AST-aware source code parsing. |
| | S3DirectoryLoader | AWS S3 | Ingests objects directly from Amazon S3 buckets. |
| | NotionDirectoryLoader | Notion export | Ingests exported Notion workspace folders. |
| | ConfluenceLoader | Confluence API | Fetches pages and attachments from Confluence. |
| | GoogleDriveLoader | Google Drive | Downloads and parses Google Docs, Sheets, and PDFs. |

### 1. Synchronous vs Lazy Loading (load vs lazy_load)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("attention.pdf")

# 1. load(): Returns a full list of Document objects in memory
docs_list = loader.load()
print(f"load() returned a {type(docs_list)} containing {len(docs_list)} documents.")

# 2. lazy_load(): Returns an iterator to yield documents one-by-one
docs_iterator = loader.lazy_load()
print(f"lazy_load() returned an iterator: {type(docs_iterator)}")

# Stream documents lazily
print("\nStreaming first 2 pages lazily:")
for i, doc in enumerate(docs_iterator):
    print(f" Page {doc.metadata['page'] + 1} loaded with {len(doc.page_content)} characters.")
    if i == 1:
        break

### 2. Simple Text Loading (TextLoader)

In [ ]:
from langchain_community.document_loaders import TextLoader

# Load text file 'speech.txt' from current directory
loader = TextLoader("speech.txt", encoding="utf-8")
docs = loader.load()

print(f"Loaded documents: {len(docs)}")
print(f"Metadata: {docs[0].metadata}")
print(f"Content Preview:\n{docs[0].page_content[:200]}")

### 3. Loading Markdown Files

In [ ]:
from langchain_community.document_loaders import TextLoader, UnstructuredMarkdownLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter

# Create sample markdown file for demonstration
sample_md = """# LangChain Architecture
LangChain provides modular abstractions for LLM applications.

## Document Loaders
Document loaders ingest raw data into standardized Document objects.

### Markdown Loader
Markdown loaders preserve structural formatting like headers.
"""

with open("sample_docs.md", "w") as f:
    f.write(sample_md)

# 1. Plain text loader
loader = TextLoader("sample_docs.md")
docs = loader.load()
print("--- TextLoader Output ---")
print(docs[0].page_content[:150])

# 2. Header-aware splitting
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
splits = splitter.split_text(sample_md)

print("\n--- Header Splitter Output ---")
for split in splits:
    print(f"Metadata: {split.metadata} | Content: {split.page_content[:50]}")

### 4. Loading PDF Documents (PyPDFLoader)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Load PDF file 'attention.pdf' from current directory
loader = PyPDFLoader("attention.pdf")
pages = loader.load()

print(f"Total Pages: {len(pages)}")
print(f"Page 1 Metadata: {pages[0].metadata}")
print(f"Page 1 Content Preview:\n{pages[0].page_content[:200]}")

### 5. Loading Structured Data (UnstructuredXMLLoader)

In [ ]:
from langchain_community.document_loaders import UnstructuredXMLLoader

# Load XML file 'records.xml' from current directory
loader = UnstructuredXMLLoader("records.xml")
docs = loader.load()

print(f"Loaded XML documents: {len(docs)}")
print(f"Metadata: {docs[0].metadata}")
print(f"Content Preview:\n{docs[0].page_content[:200]}")

### 6. Directory Batch Loading (DirectoryLoader)

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Batch load all .txt files in current folder
loader = DirectoryLoader(".", glob="*.txt", loader_cls=TextLoader, show_progress=False)
docs = loader.load()

print(f"Loaded {len(docs)} text file(s):")
for doc in docs:
    print(f" - Source: {doc.metadata['source']} ({len(doc.page_content)} chars)")

### 7. Web Page Loading (WebBaseLoader)

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

# Load web page content
loader = WebBaseLoader("https://docs.langchain.com/oss/python/langchain/overview")
docs = loader.load()

print(f"Web Documents: {len(docs)}")
print(f"Metadata: {docs[0].metadata}")
print(f"Content Preview:\n{docs[0].page_content[:200].strip()}")

### 8. Online Knowledge Loaders (ArXivLoader & WikipediaLoader)

LangChain provides loaders for querying online research papers (ArXiv) and encyclopedic knowledge (Wikipedia).

In [ ]:
from langchain_community.document_loaders import ArXivLoader, WikipediaLoader

# 1. ArXivLoader: Load paper by ArXiv ID or query
arxiv_loader = ArXivLoader(query="1706.03762", load_max_docs=1)
arxiv_docs = arxiv_loader.load()

print("--- ArXivLoader Output ---")
print(f"Loaded Documents: {len(arxiv_docs)}")
print(f"Title: {arxiv_docs[0].metadata.get('Title', 'N/A')}")
print(f"Content Preview:\n{arxiv_docs[0].page_content[:200].strip()}")

# 2. WikipediaLoader: Load article by search query
wiki_loader = WikipediaLoader(query="Generative artificial intelligence", load_max_docs=1)
wiki_docs = wiki_loader.load()

print("\n--- WikipediaLoader Output ---")
print(f"Loaded Documents: {len(wiki_docs)}")
print(f"Metadata: {wiki_docs[0].metadata}")
print(f"Content Preview:\n{wiki_docs[0].page_content[:200].strip()}")

### 9. Custom Document Loader (Subclassing BaseLoader)

To ingest custom file formats or internal database systems, subclass BaseLoader from langchain_core.document_loaders and override lazy_load().

In [ ]:
from typing import Iterator
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document

class CustomLineLoader(BaseLoader):
    """Custom loader that yields one Document object per non-empty line."""
    def __init__(self, file_path: str):
        self.file_path = file_path

    def lazy_load(self) -> Iterator[Document]:
        with open(self.file_path, "r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, 1):
                cleaned = line.strip()
                if cleaned:
                    yield Document(
                        page_content=cleaned,
                        metadata={"source": self.file_path, "line": line_no}
                    )

# Instantiate and lazy load with custom loader
custom_loader = CustomLineLoader("speech.txt")
first_three_lines = []
for doc in custom_loader.lazy_load():
    first_three_lines.append(doc)
    if len(first_three_lines) == 3:
        break

print(f"Loaded {len(first_three_lines)} line documents with custom loader.")
print(f"Line 1 Metadata: {first_three_lines[0].metadata}")
print(f"Line 1 Content: {first_three_lines[0].page_content[:100]}")

### 10. Loading and Splitting in One Step (load_and_split)

BaseLoader provides load_and_split() as a shortcut to load documents and split them using a TextSplitter in a single invocation.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("speech.txt")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)

# Combined load and split operation
chunks = loader.load_and_split(text_splitter=text_splitter)

print(f"Total chunks generated via load_and_split(): {len(chunks)}")
print(f"Chunk 1 Metadata: {chunks[0].metadata}")
print(f"Chunk 1 Content Preview:\n{chunks[0].page_content[:150]}")